# Notebook 10: Caching Patterns — Redis as a Cache

**Caching is Redis's #1 use case.** The idea is simple: store frequently accessed data in Redis (fast) so you don't have to query your database (slow) every time.

```
Without cache:  App → Database (10-100ms)
With cache:     App → Redis (0.1-1ms) → Database only on miss
```

In this notebook, you'll learn the **patterns** used in production systems worldwide.

In [ ]:
import redis
import json
import time
import random

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## Why Caching Matters — A Demo

In [ ]:
def fake_database_query(user_id):
    """Simulates a slow database query."""
    time.sleep(0.5)  # 500ms — typical for a complex DB query
    return {'id': user_id, 'name': f'User_{user_id}', 'email': f'user{user_id}@test.com'}

# Without cache — every call is slow
start = time.time()
for _ in range(5):
    result = fake_database_query(1001)
total_no_cache = time.time() - start

# With cache — first call slow, rest instant
start = time.time()
for _ in range(5):
    cached = r.get('user:1001')
    if cached:
        result = json.loads(cached)
    else:
        result = fake_database_query(1001)
        r.set('user:1001', json.dumps(result), ex=60)
total_with_cache = time.time() - start

print(f"Without cache: {total_no_cache:.2f}s (5 slow queries)")
print(f"With cache:    {total_with_cache:.2f}s (1 slow query + 4 instant)")
print(f"Speedup:       {total_no_cache/total_with_cache:.1f}x faster!")

---
## Pattern 1: Cache-Aside (Lazy Loading)

The **most common** caching pattern. The application manages the cache.

```
1. App checks cache  →  HIT? Return cached data
                     →  MISS? Query DB, store in cache, return
```

**Pros:** Only caches what's actually requested. Simple.
**Cons:** First request is always slow (cache miss). Stale data until TTL expires.

In [ ]:
r.flushdb()

# Simulated database
DATABASE = {
    1001: {'name': 'Sujit', 'email': 'sujit@test.com', 'plan': 'pro'},
    1002: {'name': 'Alice', 'email': 'alice@test.com', 'plan': 'free'},
    1003: {'name': 'Bob', 'email': 'bob@test.com', 'plan': 'pro'},
}

def db_query(user_id):
    """Simulate slow database query."""
    time.sleep(0.3)
    return DATABASE.get(user_id)

def get_user(user_id, cache_ttl=60):
    """Cache-aside pattern: check cache first, then DB."""
    cache_key = f'cache:user:{user_id}'
    
    # Step 1: Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), 'HIT'
    
    # Step 2: Cache miss — query database
    user = db_query(user_id)
    if user is None:
        return None, 'MISS (not found)'
    
    # Step 3: Store in cache for next time
    r.set(cache_key, json.dumps(user), ex=cache_ttl)
    
    return user, 'MISS (cached for next time)'

# Demo
for attempt in range(3):
    start = time.time()
    user, status = get_user(1001)
    elapsed = time.time() - start
    print(f"  Attempt {attempt+1}: [{status}] {elapsed:.3f}s — {user['name']}")

---
## Pattern 2: Write-Through Cache

Write to **cache AND database simultaneously**. The cache is always up-to-date.

```
App writes → Update Cache + Update Database → Done
App reads  → Always from Cache (always fresh)
```

**Pros:** Cache is never stale.
**Cons:** Every write is slower (two writes). Caches data that might never be read.

In [ ]:
def write_through_update(user_id, updates):
    """Write-through: update both cache and DB."""
    # Update database
    if user_id in DATABASE:
        DATABASE[user_id].update(updates)
    
    # Update cache (write-through)
    cache_key = f'cache:user:{user_id}'
    r.set(cache_key, json.dumps(DATABASE[user_id]), ex=60)
    
    return DATABASE[user_id]

# Update user's plan
print("Before update:")
user, status = get_user(1001)
print(f"  Plan: {user['plan']}")

# Write-through update
updated = write_through_update(1001, {'plan': 'enterprise'})
print(f"\nAfter write-through update:")
print(f"  DB:    {DATABASE[1001]['plan']}")

# Read from cache — already updated!
cached = json.loads(r.get('cache:user:1001'))
print(f"  Cache: {cached['plan']}")
print("  Both are in sync!")

---
## Pattern 3: Write-Behind (Write-Back)

Write to **cache immediately**, sync to database **asynchronously** later.

```
App writes → Update Cache → (later) Sync to Database
```

**Pros:** Very fast writes. Great for high-write scenarios.
**Cons:** Risk of data loss if Redis crashes before sync. More complex.

This is used by systems like Facebook's cache layer.

In [ ]:
# Simplified write-behind: write to Redis, queue for DB sync
def write_behind_update(user_id, updates):
    """Update cache immediately, queue DB write for later."""
    cache_key = f'cache:user:{user_id}'
    
    # Get current data from cache
    current = json.loads(r.get(cache_key) or '{}')
    current.update(updates)
    
    # Update cache immediately (fast!)
    r.set(cache_key, json.dumps(current), ex=60)
    
    # Queue the update for async DB write
    r.rpush('db_write_queue', json.dumps({'user_id': user_id, 'data': current}))
    
    return current

def process_write_queue():
    """Process pending DB writes (would run in background worker)."""
    count = 0
    while True:
        item = r.lpop('db_write_queue')
        if item is None:
            break
        data = json.loads(item)
        # Simulate DB write
        DATABASE[data['user_id']] = data['data']
        count += 1
    return count

# Fast writes to cache
r.set('cache:user:1001', json.dumps(DATABASE[1001]), ex=60)
write_behind_update(1001, {'last_login': 'today', 'login_count': '42'})
write_behind_update(1001, {'theme': 'dark'})

print(f"Queue length: {r.llen('db_write_queue')}")

# Later: background worker syncs to DB
synced = process_write_queue()
print(f"Synced {synced} writes to database")

---
## Cache Eviction Policies

What happens when Redis runs out of memory? The **eviction policy** decides which keys to delete.

| Policy | Description |
|---|---|
| `noeviction` | Return errors when memory is full (default) |
| `allkeys-lru` | Remove **least recently used** keys (recommended for cache) |
| `volatile-lru` | LRU, but only keys with expiry set |
| `allkeys-lfu` | Remove **least frequently used** keys |
| `volatile-lfu` | LFU, but only keys with expiry |
| `allkeys-random` | Remove random keys |
| `volatile-random` | Random, but only keys with expiry |
| `volatile-ttl` | Remove keys with shortest TTL |

In [ ]:
# Check current eviction policy
policy = r.config_get('maxmemory-policy')
maxmem = r.config_get('maxmemory')

print(f"Current eviction policy: {policy}")
print(f"Max memory setting: {maxmem}")
print("\nFor a caching use case, 'allkeys-lru' is usually the best choice.")
print("It automatically removes the least recently used keys when memory is full.")

---
## Cache Stampede (Thundering Herd)

**The Problem:** A popular cached item expires. 1000 requests arrive simultaneously. They ALL see a cache miss and ALL hit the database!

**Solution:** Use a **lock** so only one request rebuilds the cache.

In [ ]:
def get_user_safe(user_id, cache_ttl=60, lock_ttl=5):
    """Cache-aside with stampede protection."""
    cache_key = f'cache:user:{user_id}'
    lock_key = f'lock:cache:user:{user_id}'
    
    # Step 1: Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), 'HIT'
    
    # Step 2: Try to acquire lock (only one request rebuilds)
    acquired = r.set(lock_key, '1', nx=True, ex=lock_ttl)
    
    if acquired:
        # We got the lock — rebuild cache
        user = db_query(user_id)
        if user:
            r.set(cache_key, json.dumps(user), ex=cache_ttl)
        r.delete(lock_key)
        return user, 'MISS (rebuilt cache)'
    else:
        # Someone else is rebuilding — wait briefly and retry
        time.sleep(0.1)
        cached = r.get(cache_key)
        if cached:
            return json.loads(cached), 'HIT (waited for rebuild)'
        return db_query(user_id), 'MISS (fallback to DB)'

# Demo
r.delete('cache:user:1001')
for i in range(3):
    user, status = get_user_safe(1001)
    print(f"  Request {i+1}: [{status}]")

---
## Cache Invalidation

**"There are only two hard things in Computer Science: cache invalidation and naming things."** — Phil Karlton

Three approaches:

In [ ]:
# 1. TTL-Based — Let the cache expire naturally
r.set('cache:product:1', json.dumps({'name': 'Widget', 'price': 9.99}), ex=300)  # 5 min
print("1. TTL-Based: Data auto-expires after 5 minutes")
print(f"   TTL: {r.ttl('cache:product:1')}s")

# 2. Event-Based — Delete cache when data changes
def update_product(product_id, updates):
    # Update database
    print(f"\n2. Event-Based: Updating product {product_id}")
    # ... db update ...
    
    # Invalidate cache
    r.delete(f'cache:product:{product_id}')
    print(f"   Cache invalidated!")

update_product(1, {'price': 12.99})
print(f"   Cache exists? {r.exists('cache:product:1')}")

# 3. Version-Based — Include version number in cache key
version = r.incr('product:1:version')
r.set(f'cache:product:1:v{version}', json.dumps({'price': 12.99}), ex=300)
print(f"\n3. Version-Based: Using cache key with version v{version}")
print(f"   Old versions naturally expire via TTL")

---
## Real-World: Complete Caching Layer

In [ ]:
class CacheLayer:
    """A complete caching layer with cache-aside pattern."""
    
    def __init__(self, redis_client, default_ttl=300):
        self.r = redis_client
        self.default_ttl = default_ttl
        self.stats = {'hits': 0, 'misses': 0}
    
    def get(self, key):
        """Get from cache."""
        val = self.r.get(f'cache:{key}')
        if val:
            self.stats['hits'] += 1
            return json.loads(val)
        self.stats['misses'] += 1
        return None
    
    def set(self, key, value, ttl=None):
        """Store in cache."""
        self.r.set(f'cache:{key}', json.dumps(value), ex=ttl or self.default_ttl)
    
    def invalidate(self, key):
        """Remove from cache."""
        self.r.delete(f'cache:{key}')
    
    def get_or_set(self, key, fetch_fn, ttl=None):
        """Cache-aside: get from cache, or fetch and cache."""
        cached = self.get(key)
        if cached is not None:
            return cached
        
        value = fetch_fn()
        if value is not None:
            self.set(key, value, ttl)
        return value
    
    def hit_ratio(self):
        total = self.stats['hits'] + self.stats['misses']
        if total == 0:
            return 0
        return self.stats['hits'] / total * 100

# Usage
r.flushdb()
cache = CacheLayer(r, default_ttl=60)

# Simulate 20 requests for 5 different users
for _ in range(20):
    user_id = random.randint(1001, 1003)
    user = cache.get_or_set(
        f'user:{user_id}',
        lambda uid=user_id: db_query(uid)
    )

print(f"Stats: {cache.stats}")
print(f"Hit ratio: {cache.hit_ratio():.1f}%")

---
## Monitoring Cache Performance

In [ ]:
# Redis tracks hits/misses at the server level
stats = r.info('stats')

hits = stats['keyspace_hits']
misses = stats['keyspace_misses']
total = hits + misses

print(f"Server-wide cache stats:")
print(f"  Hits:      {hits}")
print(f"  Misses:    {misses}")
print(f"  Hit ratio: {(hits/total*100) if total > 0 else 0:.1f}%")
print(f"\nA healthy cache typically has 90%+ hit ratio.")
print(f"Below 80% suggests keys are expiring too fast or not being cached.")

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

| Pattern | How It Works | Best For |
|---|---|---|
| **Cache-Aside** | Check cache → miss → query DB → cache result | Most common, general purpose |
| **Write-Through** | Write to cache + DB together | When cache must always be fresh |
| **Write-Behind** | Write to cache now, DB later | High-write workloads |

| Concept | Details |
|---|---|
| **Eviction** | `allkeys-lru` for caching use cases |
| **Stampede** | Use SET NX lock to prevent thundering herd |
| **Invalidation** | TTL-based, event-based, or version-based |
| **Monitoring** | Track hit ratio with `INFO stats` |

---
## Exercises

1. **Cache Decorator:** Write a Python decorator `@redis_cache(ttl=30)` that automatically caches any function's return value. The cache key should be based on the function name and arguments.

2. **Multi-Tier Cache:** Implement a two-level cache: check Python dict (L1) → Redis (L2) → Database. Compare performance.

3. **Cache Warming:** Write a function that pre-loads the 10 most popular items into the cache on application startup.

4. **Stale-While-Revalidate:** Implement a pattern where slightly stale data is returned immediately while a background thread refreshes the cache.

5. **Cache Analytics:** Build a system that tracks per-key hit/miss rates and identifies the most and least useful cached keys.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 11 — Streams](./11_Streams.ipynb)** — Event streaming, consumer groups, and building robust message queues!